# SwarmSort Gen 2, notebook 7: Gen 1 vs Gen 2 on the same test images (6 shared classes)

**What:** scores all six final models, Gen 1 arms A, B and C and Gen 2 arms A, B and C, on the SAME test images with
`src/evaluate.py --arms configs/gen2/arms_gen1_vs_gen2.yaml`, twice:

- `g1g2_real_world`: the 468 real-world test photos (TACO + HITL), which Gen 1 never saw in training;
- `g1g2_studio`: the 1046 studio test photos (Gen 1's own test split, which neither generation trained on).

**Why:** this is the headline result of Gen 2. Did training with real-world data beat Gen 1 on real-world photos,
and what did it cost on studio photos? The comparison is made fair in three ways:

1. **Same classes.** Gen 1 knows 6 classes, Gen 2 also has OTHER. The test labels are restricted to the 6 shared
   classes: `src/compare_generations.py subset` writes a copy of each test set with the OTHER label lines removed
   (images that only had OTHER stay in, with no objects, for every model). Gen 2's OTHER predictions are dropped with
   `--classes 0 1 2 3 4 5` (Ultralytics' classes filter in the NMS).
2. **Same images.** Every model is scored on the same files, so the paired bootstrap compares them image by image.
3. **Each model as deployed.** Each model runs at its own training image size (Gen 1 416, Gen 2 640); the report
   header says so.

The test images were already scored once in `g2_6_evaluate`. Here the same fixed, final models are compared on them;
nothing is trained or tuned on these numbers.

**Inputs** (Add Input -> Notebook Output, the finished version of each):

| Notebook output | Gives |
|---|---|
| `g2_0_build_dataset` | the test images (`gen2_data/`) |
| `g2_1_arm_a` | `gen2_arm_a_best.pt` |
| `g2_4_arm_b` | `gen2_arm_b_best.pt` |
| `g2_5_arm_c` | `gen2_arm_c_best.pt` |

The Gen 1 weights come with the repository clone (`app/models/arm_{a,b,c}.pt`); the configs and training JSONs of
both generations are committed too. Every weights file is checked against the sha256 recorded when it was scored on
its own test set (`results/test_results.json`, `results/gen2/test_all_results.json`), so the right model is in each
slot. **Accelerator: GPU T4 x2**, **Internet: on**. Run it with **Save Version -> Save & Run All (Commit)**.

**Expected time:** about 10-15 minutes (2 evaluation runs of 6 models each: validation, latency, 1000 paired
bootstrap resamples, sample predictions). The log prints a `[time]` line at the end of each phase of each run.

**Let the commit finish.** Cancelling a committed version loses all of its output (nothing is saved), so do not stop
it early.

**Freeze protection** (as in g2_6). The attached dataset is first copied to local disk, and every script runs under a
watchdog that restarts it after 20 minutes without output or file changes (at most 3 times).

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
assert os.path.isdir(f"{REPO}/src"), "git clone failed: turn on Settings -> Internet and run again"
%cd {REPO}
# Kaggle's own CUDA build of torch is kept: only Ultralytics is installed, at the version pinned in requirements.txt
!pip install -q ultralytics==8.4.169
import torch, ultralytics
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU T4 x2 and run again"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))
if not os.path.isfile("src/watchdog_run.py"):
    raise RuntimeError("src/watchdog_run.py is not on GitHub yet: push the latest commits")
sys.path.insert(0, f"{REPO}/src")
from watchdog_run import run_watched
# About 10 s: check on this machine that a frozen script and its worker process are killed and retried
subprocess.run([sys.executable, "-m", "unittest", "-q",
                "test_watchdog.TestWatchdog.test_b_stalled_script_is_killed_and_retried",
                "test_watchdog.TestWatchdog.test_e_child_processes_are_killed_too"], cwd="tests", check=True)


def run(*args, watch=(), stall_minutes=20, max_retries=3):
    """Run a repository script under the stall watchdog (src/watchdog_run.py). Its output is streamed here. If it
    prints nothing and nothing changes under `watch` for stall_minutes, it is killed together with its dataloader
    workers and started again with the same arguments, at most max_retries times (search.py then continues from its
    log, train_final.py --resume from last.pt). If the script stops with an error, this cell fails and the version
    is marked failed (as with check=True)."""
    args = [str(a) for a in args]
    print("$ python", " ".join(args), flush=True)
    run_watched([sys.executable, *args], watch=watch, stall_minutes=stall_minutes, max_retries=max_retries,
                env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
# Find the dataset built by g2_0_build_dataset (attached as input), copy it to this machine's local disk, and write
# configs that point at the local copy: /kaggle/working/gen2_cfg/data.yaml, data_test_<domain>.yaml, split.csv, lists/
# Why the copy: two Gen 2 runs froze about 55 minutes into training and sat idle until Kaggle's 12-hour limit,
# most likely because the attached notebook-output mount under /kaggle/input stopped answering. Training now reads
# only the local copy. It goes outside /kaggle/working, so it does not become part of this version's output.
# The copy reads that mount too, so it fails loudly (TimeoutError) if no file arrives for 2 minutes.
import glob, json, shutil
from watchdog_run import copy_files, list_tree
INPUT, CFG = "/kaggle/input", "/kaggle/working/gen2_cfg"
builds = sorted({os.path.dirname(p) for p in glob.glob(f"{INPUT}/**/build_report.json", recursive=True)
                 if os.path.isfile(os.path.join(os.path.dirname(p), "data.yaml"))
                 and os.path.isdir(os.path.join(os.path.dirname(p), "images"))})
if not builds:
    raise FileNotFoundError("The Generation 2 dataset is not attached. Add Input -> Notebook Output -> "
                            "g2_0_build_dataset (its finished version), then run again.")
if len(builds) > 1:
    raise RuntimeError("Several built datasets are attached; remove all but one version of g2_0_build_dataset:\n  "
                       + "\n  ".join(builds))
BUILD_SRC = builds[0]
print("Built dataset (attached):", BUILD_SRC)
files = list_tree(BUILD_SRC, exclude=["*.cache"])
size = sum(n for _, n in files)
for base in ("/kaggle/tmp", "/kaggle/temp", "/tmp"):  # local disk outside /kaggle/working
    if os.path.isdir(base) and os.access(base, os.W_OK) and shutil.disk_usage(base).free > 1.1 * size + 2e9:
        BUILD = f"{base}/gen2_data"
        break
else:
    raise OSError(f"No local folder has {size / 1e9:.1f} GB free for the dataset copy")
stamp = {"source": BUILD_SRC, "files": len(files), "bytes": size}
if os.path.isfile(f"{BUILD}/.copied.json") and json.load(open(f"{BUILD}/.copied.json")) == stamp:
    print("Local copy already made in this session:", BUILD)
else:
    shutil.rmtree(BUILD, ignore_errors=True)
    print(f"Copying {len(files)} files ({size / 1e9:.2f} GB) to {BUILD}", flush=True)
    info = copy_files(BUILD_SRC, BUILD, files, workers=16, stall_seconds=120)
    if list_tree(BUILD) != files:
        raise RuntimeError(f"The local copy {BUILD} differs from {BUILD_SRC} (file names or sizes)")
    json.dump(stamp, open(f"{BUILD}/.copied.json", "w"))
    print(f"Copied {info['files']} files, {info['bytes'] / 1e9:.2f} GB in {info['seconds']:.0f} s "
          f"({info['bytes'] / 1e6 / max(info['seconds'], 1):.0f} MB/s); names and sizes match the attached build")
for sub in ("images", "labels"):
    print(f"  {sub}/: {len(os.listdir(f'{BUILD}/{sub}'))} files")
if "--relocate-from" not in open("src/build_dataset.py", encoding="utf-8").read():
    raise RuntimeError("src/build_dataset.py on GitHub has no --relocate-from mode yet: push the latest commits")
run("src/build_dataset.py", "--relocate-from", BUILD, "--out", CFG)
assert os.path.isfile(f"{CFG}/data.yaml"), f"the relocation did not write {CFG}/data.yaml"
print(open(f"{CFG}/data.yaml").read())
print("Test-set yamls:", sorted(os.path.basename(p) for p in glob.glob(f"{CFG}/data_test_*.yaml")))


In [ ]:
# The six models. Gen 1: app/models/arm_<x>.pt from the clone. Gen 2: gen2_arm_<x>_best.pt from the attached outputs
# (or app/models/gen2/arm_<x>.pt if the clone has it), copied to the paths in configs/gen2/arms_gen1_vs_gen2.yaml.
# Each file's sha256 must equal the sha256_12 recorded when that model was scored on its own test set
# (Gen 1: results/test_results.json, Gen 2: results/gen2/test_all_results.json).
import hashlib, shutil, yaml
from collections import defaultdict
ARMS_YAML = "configs/gen2/arms_gen1_vs_gen2.yaml"
for f in (ARMS_YAML, "src/compare_generations.py"):
    if not os.path.isfile(f):
        raise FileNotFoundError(f"{f} is not on GitHub yet: push the latest commits")
if "--classes" not in open("src/evaluate.py", encoding="utf-8").read():
    raise RuntimeError("src/evaluate.py on GitHub has no --classes option yet: push the latest commits")
arms = yaml.safe_load(open(ARMS_YAML))["arms"]
RECORDED = {"G1": "results/test_results.json", "G2": "results/gen2/test_all_results.json"}
needed = [*RECORDED.values(), *(arm[k] for arm in arms.values() for k in ("config", "val_json") if arm.get(k))]
missing = [f for f in needed if not os.path.isfile(f)]
if missing:
    raise FileNotFoundError(f"not in the clone: {missing}; commit them and push")
recorded = {g: {label: m["sha256_12"] for label, m in json.load(open(p))["models"].items()} for g, p in RECORDED.items()}
NOTEBOOK = {"a": "g2_1_arm_a", "b": "g2_4_arm_b", "c": "g2_5_arm_c"}
index = defaultdict(list)  # file name -> every path under /kaggle/input (image and label folders skipped)
for root, dirs, files in os.walk(INPUT):
    dirs[:] = [d for d in dirs if d not in ("images", "labels")]
    for f in files:
        index[f].append(os.path.join(root, f).replace(os.sep, "/"))


def sha12(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:12]


rows = []
for label, arm in arms.items():
    gen, x = label[:2], label[2].lower()
    want = recorded[gen][x.upper()]
    if gen == "G1":
        if not os.path.isfile(arm["model"]):
            raise FileNotFoundError(f"{arm['model']} is not in the clone: commit app/models/ and push")
        src = "(repository)"
    else:
        candidates = sorted(index[f"gen2_arm_{x}_best.pt"], key=len)  # attached outputs first, top-level copy first
        candidates += [p for p in (f"app/models/gen2/arm_{x}.pt",) if os.path.isfile(p)]
        if not candidates:
            raise FileNotFoundError(f"No weights for {label} (gen2_arm_{x}_best.pt) under {INPUT}: Add Input -> "
                                    f"Notebook Output -> {NOTEBOOK[x]} (its finished version), then run again.")
        good = [p for p in candidates if sha12(p) == want]
        for p in candidates:
            if p not in good:
                print(f"Note: {p} is not the {label} model scored in g2_6 (sha256 {sha12(p)}, expected {want}); "
                      f"not used")
        if not good:
            raise RuntimeError(f"None of the weights found for {label} has the sha256 {want} recorded in "
                               f"{RECORDED['G2']}: attach the version of {NOTEBOOK[x]} that g2_6 scored.")
        src = good[0]
        os.makedirs(os.path.dirname(arm["model"]), exist_ok=True)
        shutil.copyfile(src, arm["model"])
    got = sha12(arm["model"])
    if got != want:
        raise RuntimeError(f"{arm['model']} has sha256 {got}, but {label} was scored with {want} ({RECORDED[gen]})")
    rows.append((label, arm["name"], arm["model"], got, src))
print(f"{'arm':<5}{'name':<22}{'model':<42}{'sha256':<14}from")
for row in rows:
    print(f"{row[0]:<5}{row[1]:<22}{row[2]:<42}{row[3]:<14}{row[4]}")


In [ ]:
# The 6-class test sets: the same images, with the OTHER (class 6) label lines removed. Images that only had OTHER
# stay in, as images with no shared-class objects (the same for every model). The derived sets go next to the local
# dataset copy (outside /kaggle/working, so they are not part of the output); the images are hard links, not copies.
names = yaml.safe_load(open(f"{CFG}/data.yaml"))["names"]
assert len(names) == 7 and names[6] == "OTHER", f"expected the 7 Gen 2 classes with OTHER last, got {names}"
WORK = os.path.dirname(BUILD)
SETS = {}
for d in ("real_world", "studio"):
    src = f"{CFG}/data_test_{d}.yaml"
    if not os.path.isfile(src):
        raise FileNotFoundError(f"{src} missing: the build has no {d} test set (see g2_0_build_dataset)")
    out = f"{WORK}/g1g2_{d}"
    run("src/compare_generations.py", "subset", "--data", src, "--split", "test", "--classes", "0-5", "--out", out)
    rep = json.load(open(f"{out}/subset_report.json"))
    print(f"  {d}: {rep['images']} images, {rep['boxes_kept']} boxes of the 6 classes, {rep['boxes_removed']} OTHER "
          f"boxes removed, {rep['images_emptied']} images left with no objects\n")
    SETS[d] = f"{out}/data.yaml"


In [ ]:
# The six models on each 6-class test set. evaluate.py first checks every checkpoint (Gen 1: 6 classes, 100 epochs,
# imgsz 416; Gen 2: class names starting with the 6, 100 epochs, imgsz 640; each arm's hyperparameters), then
# validates each model at its own imgsz with --classes 0..5, times it, and runs the paired bootstrap.
TAGS = []
for d, data in SETS.items():
    tag = f"g1g2_{d}"
    print(f"\n=== test split, {tag} ===")
    run("src/evaluate.py", "--arms", ARMS_YAML, "--data", data, "--split", "test", "--classes", 0, 1, 2, 3, 4, 5,
        "--out-dir", "results/gen2", "--tag", tag, "--device", 0, watch=["results/gen2"])
    TAGS.append(tag)


In [ ]:
# The comparison reports written by evaluate.py
from IPython.display import Markdown, display
for tag in TAGS:
    display(Markdown(open(f"results/gen2/test_{tag}_comparison.md", encoding="utf-8").read()))


In [ ]:
# Headline: all six arms on both test sets, and the key verdicts (G2X vs G1X per arm letter; best Gen 2 vs best Gen 1,
# each chosen by its own validation mAP@50, not by these test numbers). Also saved as results/gen2/test_g1g2_summary.md
run("src/compare_generations.py", "summary", "--results", *[f"results/gen2/test_{t}_results.json" for t in TAGS],
    "--out", "results/gen2/test_g1g2_summary.md")


### Files to download / where they go locally

From the **Output** tab of the finished version (all under `SwarmSort/results/gen2/`):

| File in the Output tab | Where it goes locally |
|---|---|
| `test_g1g2_real_world_results.json`, `test_g1g2_studio_results.json` | `results/gen2/` |
| `test_g1g2_real_world_comparison.md`, `test_g1g2_studio_comparison.md` | `results/gen2/` |
| `test_g1g2_summary.md` (the headline table and key verdicts) | `results/gen2/` |
| `plots/test_g1g2_*` (confusion matrices, per-class AP, sample predictions) | `results/gen2/plots/` |

Commit them all. The copied Gen 2 weights and the `runs/` folder are not needed (the weights are already in the
Kaggle outputs of notebooks 1, 4 and 5).

### If the session dies

Run it again with the same inputs: the evaluation is deterministic (seeded bootstrap), so it gives the same numbers,
and nothing needs to be carried over.